# Tahap 4: Text Representation

Notebook ini merepresentasikan blok abu-abu **Text Representation** pada diagram metodologi penelitian, yang terdiri dari 3 sub-tahapan representasi fitur sesuai kebutuhan masing-masing algoritma:
1. **Bag-of-Words (BoW)** $\rightarrow$ Digunakan untuk model **LDA**
2. **Semantic Vector Space (Joint Word-Document Embedding)** $\rightarrow$ Digunakan untuk model **Top2Vec**
3. **Transformer Embedding (Contextual Sentence-BERT)** $\rightarrow$ Digunakan untuk model **BERTopic**

In [ ]:
import os
import ast
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sentence_transformers import SentenceTransformer

os.makedirs('representations', exist_ok=True)

## 4.0 Memuat Data Hasil *Text Preprocessing* (`03_preprocessed_data.csv`)

In [ ]:
df = pd.read_csv('03_preprocessed_data.csv')
df['tokens'] = df['step5_stemmed_tokens'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
texts = df['text_preprocessed'].astype(str).tolist()
tokenized_docs = df['tokens'].tolist()

print(f'Jumlah dokumen siap direpresentasikan: {len(texts)}')
print('Contoh dokumen:', texts[0])
print('Contoh token  :', tokenized_docs[0])

## 4.1 Representasi 1: Bag-of-Words (BoW) untuk LDA
Membentuk kamus kata (*Gensim Dictionary*) dan matriks frekuensi kata (*Document-Term Matrix* / BoW Corpus) dengan memfilter kata yang muncul terlalu jarang (`no_below=5`) atau terlalu dominan (`no_above=0.7`).

In [ ]:
# 1. Gensim Dictionary & BoW Corpus (untuk Gensim LDA & Coherence Evaluation)
gensim_dict = Dictionary(tokenized_docs)
gensim_dict.filter_extremes(no_below=5, no_above=0.7)
bow_corpus = [gensim_dict.doc2bow(doc) for doc in tokenized_docs]

# 2. Scikit-Learn CountVectorizer (Matriks BoW)
count_vectorizer = CountVectorizer(min_df=5, max_df=0.7)
bow_matrix = count_vectorizer.fit_transform(texts)

print(f'Ukuran Kosakata (Vocabulary) Gensim BoW : {len(gensim_dict)} kata unik')
print(f'Dimensi Matriks BoW (Dokumen x Kata)    : {bow_matrix.shape}')

# Simpan representasi Bag-of-Words
with open('representations/bow_representation.pkl', 'wb') as f:
    pickle.dump({
        'gensim_dict': gensim_dict,
        'bow_corpus': bow_corpus,
        'count_vectorizer': count_vectorizer,
        'bow_matrix': bow_matrix,
        'tokenized_docs': tokenized_docs,
        'texts': texts
    }, f)
print('Representasi Bag-of-Words disimpan ke: representations/bow_representation.pkl')

## 4.2 Representasi 2: Semantic Vector Space (Joint Word-Document Embedding) untuk Top2Vec
Membangun ruang vektor semantik bersama (*Joint Document & Word Semantic Vector Space*) menggunakan arsitektur Distributed Bag of Words / Skip-Gram (`Doc2Vec`) di mana dokumen dan kata berada pada ruang dimensi vektor yang sama ($\mathbb{R}^{100}$).

In [ ]:
tagged_docs = [TaggedDocument(words=doc, tags=[i]) for i, doc in enumerate(tokenized_docs)]

# Melatih Joint Word-Document Semantic Space
d2v_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=100,
    window=10,
    min_count=5,
    workers=4,
    dm=0,           # Distributed Bag of Words (DBOW) + Skip-gram word vectors (sesuai paper Top2Vec)
    dbow_words=1,   # Melatih word vectors dan document vectors secara bersamaan di ruang yang sama
    epochs=40,
    seed=42
)

doc_vectors_semantic = np.array([d2v_model.dv[i] for i in range(len(tokenized_docs))])
word_vocab_semantic = list(d2v_model.wv.index_to_key)
word_vectors_semantic = np.array([d2v_model.wv[w] for w in word_vocab_semantic])

print(f'Dimensi Document Vectors (Semantic Space) : {doc_vectors_semantic.shape}')
print(f'Dimensi Word Vectors (Semantic Space)     : {word_vectors_semantic.shape}')

with open('representations/semantic_vector_space.pkl', 'wb') as f:
    pickle.dump({
        'd2v_model': d2v_model,
        'doc_vectors': doc_vectors_semantic,
        'word_vocab': word_vocab_semantic,
        'word_vectors': word_vectors_semantic
    }, f)
print('Representasi Semantic Vector Space disimpan ke: representations/semantic_vector_space.pkl')

## 4.3 Representasi 3: Transformer Embedding untuk BERTopic
Mengekstrak *Contextualized Document Embeddings* menggunakan model *Pre-trained Multilingual Sentence-Transformer* (`paraphrase-multilingual-MiniLM-L12-v2`) yang mendukung Bahasa Indonesia.

In [ ]:
embedding_model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
sentence_model = SentenceTransformer(embedding_model_name)

transformer_embeddings = sentence_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f'Dimensi Transformer Embeddings : {transformer_embeddings.shape}')

np.save('representations/transformer_embeddings.npy', transformer_embeddings)
with open('representations/transformer_meta.pkl', 'wb') as f:
    pickle.dump({'model_name': embedding_model_name}, f)

print('Representasi Transformer Embedding disimpan ke: representations/transformer_embeddings.npy')